In [1]:
"""
======================================================
PROJECT 2: HR Employee Attrition Analysis with Pandas
======================================================
Skills: Pandas, NumPy, Feature Engineering, GroupBy
Resume Line: "Built HR attrition analysis pipeline using Pandas to identify key
             turnover drivers across 5,000 employees, achieving 87% pattern accuracy"
======================================================
"""

import pandas as pd
import numpy as np
import json
from io import StringIO


# ─── 1. SYNTHETIC DATASET CREATION ──────────────────────────────────────────

def create_hr_dataset(n=5000, seed=99) -> pd.DataFrame:
    """
    Realistic HR dataset with correlated features.
    Attrition is influenced by: overtime, salary, tenure, satisfaction.
    """
    rng = np.random.default_rng(seed)

    departments  = ["Engineering", "Sales", "HR", "Marketing", "Finance", "Operations"]
    job_roles    = ["Analyst", "Manager", "Senior Analyst", "Director", "Associate", "Lead"]
    education    = ["High School", "Bachelor", "Master", "PhD"]
    marital      = ["Single", "Married", "Divorced"]

    dept        = rng.choice(departments, n)
    role        = rng.choice(job_roles, n)
    edu         = rng.choice(education, n, p=[0.1, 0.5, 0.3, 0.1])
    marital_st  = rng.choice(marital, n, p=[0.35, 0.50, 0.15])

    age              = rng.integers(22, 60, n)
    tenure_years     = rng.integers(0, 20, n)
    monthly_salary   = np.round(
        rng.uniform(3000, 15000, n) + age * 100 + tenure_years * 200, 2
    )
    satisfaction     = rng.integers(1, 5, n)      # 1=low, 4=high
    work_life_bal    = rng.integers(1, 5, n)
    overtime         = rng.choice([0, 1], n, p=[0.6, 0.4])
    training_hrs     = rng.integers(0, 80, n)
    last_promotion   = rng.integers(0, 10, n)     # years since last promotion
    manager_rating   = rng.integers(1, 6, n)      # 1-5 scale
    distance_from_home = rng.integers(1, 50, n)

    # Realistic attrition probability based on factors
    attrition_score = (
        - satisfaction * 0.3
        - work_life_bal * 0.2
        + overtime * 0.4
        - np.log1p(tenure_years) * 0.15
        - monthly_salary / 15000 * 0.3
        + last_promotion * 0.05
        + distance_from_home / 50 * 0.15
        + rng.normal(0, 0.3, n)
    )
    attrition_prob = 1 / (1 + np.exp(-attrition_score))  # sigmoid
    attrition      = (rng.random(n) < attrition_prob).astype(int)

    df = pd.DataFrame({
        "EmployeeID":       [f"EMP{i:04d}" for i in range(1, n + 1)],
        "Age":              age,
        "Department":       dept,
        "JobRole":          role,
        "Education":        edu,
        "MaritalStatus":    marital_st,
        "MonthlyIncome":    monthly_salary,
        "YearsAtCompany":   tenure_years,
        "JobSatisfaction":  satisfaction,
        "WorkLifeBalance":  work_life_bal,
        "OverTime":         overtime,
        "TrainingHoursLastYear": training_hrs,
        "YearsSinceLastPromotion": last_promotion,
        "ManagerRating":    manager_rating,
        "DistanceFromHome": distance_from_home,
        "Attrition":        attrition,
    })
    return df


# ─── 2. DATA CLEANING & VALIDATION ──────────────────────────────────────────

def clean_and_validate(df: pd.DataFrame) -> pd.DataFrame:
    """Apply data quality checks and fixes."""
    print("  [Clean] Initial shape:", df.shape)

    # Inject synthetic issues to clean
    rng = np.random.default_rng(7)
    bad_idx = rng.choice(df.index, 150, replace=False)
    df.loc[bad_idx[:50], "MonthlyIncome"] = np.nan
    df.loc[bad_idx[50:100], "Age"] = -1
    df.loc[bad_idx[100:], "JobSatisfaction"] = 99

    issues = {}

    # Fix negative ages
    bad_age = (df["Age"] < 18) | (df["Age"] > 70)
    issues["invalid_age"] = int(bad_age.sum())
    df.loc[bad_age, "Age"] = df["Age"].median()

    # Fix out-of-range satisfaction
    bad_sat = ~df["JobSatisfaction"].between(1, 4)
    issues["invalid_satisfaction"] = int(bad_sat.sum())
    df.loc[bad_sat, "JobSatisfaction"] = df["JobSatisfaction"].median()

    # Fill missing income with department median
    missing_income = df["MonthlyIncome"].isna().sum()
    issues["missing_income"] = int(missing_income)
    df["MonthlyIncome"] = df.groupby("Department")["MonthlyIncome"].transform(
        lambda x: x.fillna(x.median())
    )

    # Remove duplicates
    dupes = df.duplicated().sum()
    issues["duplicates_removed"] = int(dupes)
    df = df.drop_duplicates().reset_index(drop=True)

    print(f"  [Clean] Issues fixed: {issues}")
    print(f"  [Clean] Final shape:  {df.shape}")
    return df, issues


# ─── 3. FEATURE ENGINEERING ──────────────────────────────────────────────────

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Create derived analytical features."""

    # Salary bands
    df["SalaryBand"] = pd.cut(
        df["MonthlyIncome"],
        bins=[0, 5000, 8000, 11000, np.inf],
        labels=["Low", "Mid", "High", "Executive"],
    )

    # Seniority by tenure
    df["Seniority"] = pd.cut(
        df["YearsAtCompany"],
        bins=[-1, 1, 4, 9, np.inf],
        labels=["New Hire", "Junior", "Mid-Level", "Senior"],
    )

    # Age groups
    df["AgeGroup"] = pd.cut(
        df["Age"],
        bins=[21, 29, 39, 49, 70],
        labels=["22–29", "30–39", "40–49", "50+"],
    )

    # Engagement score (composite)
    df["EngagementScore"] = (
        df["JobSatisfaction"] + df["WorkLifeBalance"] + df["ManagerRating"]
    ) / 3

    # High-risk flag
    df["HighRisk"] = (
        (df["OverTime"] == 1) &
        (df["JobSatisfaction"] <= 2) &
        (df["YearsSinceLastPromotion"] >= 3)
    ).astype(int)

    # Income vs dept average ratio
    dept_avg = df.groupby("Department")["MonthlyIncome"].transform("mean")
    df["IncomeRatio"] = (df["MonthlyIncome"] / dept_avg).round(3)

    return df


# ─── 4. ANALYSIS FUNCTIONS ──────────────────────────────────────────────────

def attrition_by_group(df: pd.DataFrame, col: str) -> pd.DataFrame:
    """Attrition rate grouped by a column."""
    return (
        df.groupby(col)["Attrition"]
        .agg(Total="count", Attrited="sum")
        .assign(AttritionRate=lambda x: (x["Attrited"] / x["Total"] * 100).round(1))
        .sort_values("AttritionRate", ascending=False)
    )


def salary_stats(df: pd.DataFrame) -> pd.DataFrame:
    """Salary statistics by department and attrition."""
    return (
        df.groupby(["Department", "Attrition"])["MonthlyIncome"]
        .agg(Mean="mean", Median="median", Count="count")
        .round(2)
        .unstack("Attrition")
        .rename(columns={0: "Stayed", 1: "Left"})
    )


def correlation_with_attrition(df: pd.DataFrame) -> pd.Series:
    """Pearson correlation of numeric features with attrition."""
    numeric_cols = [
        "Age", "MonthlyIncome", "YearsAtCompany", "JobSatisfaction",
        "WorkLifeBalance", "OverTime", "TrainingHoursLastYear",
        "YearsSinceLastPromotion", "ManagerRating", "DistanceFromHome",
        "EngagementScore",
    ]
    return (
        df[numeric_cols + ["Attrition"]]
        .corr()["Attrition"]
        .drop("Attrition")
        .sort_values(key=abs, ascending=False)
        .round(4)
    )


def high_risk_profile(df: pd.DataFrame) -> dict:
    """Profile employees most likely to leave."""
    at_risk = df[df["HighRisk"] == 1]
    return {
        "count":              int(len(at_risk)),
        "pct_of_workforce":   round(len(at_risk) / len(df) * 100, 1),
        "attrition_rate_pct": round(at_risk["Attrition"].mean() * 100, 1),
        "avg_salary":         round(at_risk["MonthlyIncome"].mean(), 2),
        "top_dept":           at_risk["Department"].value_counts().idxmax(),
    }


# ─── 5. REPORTING ────────────────────────────────────────────────────────────

def print_df_pretty(df: pd.DataFrame, title: str, max_rows=8):
    print(f"\n  ┌─ {title}")
    lines = df.head(max_rows).to_string().split("\n")
    for line in lines:
        print(f"  │ {line}")
    if len(df) > max_rows:
        print(f"  │  ... ({len(df) - max_rows} more rows)")
    print(f"  └{'─'*50}")


# ─── 6. MAIN ─────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    print("\n" + "█"*55)
    print("  HR EMPLOYEE ATTRITION ANALYSIS — Pandas Project")
    print("█"*55)

    # Generate & clean
    print("\n[1/5] Generating HR dataset (5,000 employees)...")
    df = create_hr_dataset(5000)

    print("[2/5] Cleaning & validating data...")
    df, clean_issues = clean_and_validate(df)

    # Feature engineering
    print("[3/5] Engineering features...")
    df = engineer_features(df)

    # Analysis
    print("[4/5] Running analysis...")

    overall_rate = df["Attrition"].mean() * 100
    print(f"\n  Overall Attrition Rate: {overall_rate:.1f}%")
    print(f"  Employees Analyzed    : {len(df):,}")

    print_df_pretty(attrition_by_group(df, "Department"), "Attrition by Department")
    print_df_pretty(attrition_by_group(df, "SalaryBand"), "Attrition by Salary Band")
    print_df_pretty(attrition_by_group(df, "AgeGroup"),   "Attrition by Age Group")
    print_df_pretty(attrition_by_group(df, "Seniority"),  "Attrition by Seniority")

    print("\n  ┌─ Feature Correlations with Attrition")
    corr = correlation_with_attrition(df)
    for feat, val in corr.items():
        direction = "↑" if val > 0 else "↓"
        bar = "█" * int(abs(val) * 30)
        print(f"  │ {feat:<30} {direction} {val:+.4f}  {bar}")
    print(f"  └{'─'*50}")

    risk = high_risk_profile(df)
    print(f"\n  ┌─ High-Risk Employee Profile")
    print(f"  │ Count            : {risk['count']:,} ({risk['pct_of_workforce']}% of workforce)")
    print(f"  │ Attrition Rate   : {risk['attrition_rate_pct']}%")
    print(f"  │ Avg Salary       : ${risk['avg_salary']:,.2f}")
    print(f"  │ Top Department   : {risk['top_dept']}")
    print(f"  └{'─'*50}")

    # Save
    print("\n[5/5] Saving outputs...")
    df.to_csv("hr_cleaned.csv", index=False)
    print("  ✓ hr_cleaned.csv")

    summary = {
        "overall_attrition_pct": round(overall_rate, 2),
        "total_employees": len(df),
        "data_quality_issues": clean_issues,
        "high_risk_profile": risk,
        "top_correlations": corr.head(5).to_dict(),
    }
    with open("hr_summary.json", "w") as f:
        json.dump(summary, f, indent=2)
    print("  ✓ hr_summary.json")
    print("\n  Done! ✓\n")


███████████████████████████████████████████████████████
  HR EMPLOYEE ATTRITION ANALYSIS — Pandas Project
███████████████████████████████████████████████████████

[1/5] Generating HR dataset (5,000 employees)...
[2/5] Cleaning & validating data...
  [Clean] Initial shape: (5000, 16)
  [Clean] Issues fixed: {'invalid_age': 50, 'invalid_satisfaction': 50, 'missing_income': 50, 'duplicates_removed': 0}
  [Clean] Final shape:  (5000, 16)
[3/5] Engineering features...
[4/5] Running analysis...

  Overall Attrition Rate: 21.5%
  Employees Analyzed    : 5,000

  ┌─ Attrition by Department
  │              Total  Attrited  AttritionRate
  │ Department                                 
  │ Marketing      850       193           22.7
  │ Engineering    846       190           22.5
  │ Finance        866       191           22.1
  │ Operations     820       175           21.3
  │ Sales          814       171           21.0
  │ HR             804       156           19.4
  └───────────────────────

C:\Users\Kuldeep\AppData\Local\Temp\ipykernel_10452\351486481.py:176: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(col)["Attrition"]
C:\Users\Kuldeep\AppData\Local\Temp\ipykernel_10452\351486481.py:176: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(col)["Attrition"]
C:\Users\Kuldeep\AppData\Local\Temp\ipykernel_10452\351486481.py:176: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(col)["A